#  RT-DETR Pothole Detection
**Real-Time DEtection TRansformer** fine-tuned for pothole detection



In [1]:
# Installs ultralytics, kagglehub, seaborn and tqdm.
import subprocess, sys


def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])


pip_install("ultralytics", "kagglehub", "seaborn", "tqdm")
print("Dependencies installed")

Dependencies installed



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/saved_models"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
    SAVE_DIR = "/content/saved_models"
else:
    print("Running on LOCAL JUPYTER")
    ROOT = "."
    SAVE_DIR = "./saved_models"

OUTPUT_DIR = os.path.join(ROOT, "rtdetr_results")
DATA_DIR = os.path.join(ROOT, "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"ROOT       : {ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"SAVE_DIR   : {SAVE_DIR}")

Running on LOCAL JUPYTER
ROOT       : .
OUTPUT_DIR : ./rtdetr_results
SAVE_DIR   : ./saved_models


In [3]:
# Loads all shared imports and confirms GPU availability.
import torch, numpy as np, pandas as pd, cv2
import time, json, glob, warnings, shutil, yaml
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device      : {DEVICE}")
print(f"Torch       : {torch.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"OpenCV      : {cv2.__version__}")

Device      : cuda
Torch       : 2.10.0+cu128
NumPy       : 2.2.6
OpenCV      : 4.13.0


In [4]:
# Defines the fixed evaluation config, CLASS_NAMES, CONF_THRESH, IOU_THRESH, IMG_SIZE, and the SKIP_TRAINING switch.
CLASS_NAMES = ["pothole"]
CONF_THRESH = 0.25
IOU_THRESH = 0.45
IMG_SIZE = 640
MAX_IMAGES = None
SKIP_TRAINING = True  # True = load saved weights, False = train

PALETTE = {
    "YOLOv8m": "#00d4ff",
    "YOLOv10m": "#3b82f6",
    "YOLOv11m": "#6366f1",
    "Faster R-CNN": "#f97316",
    "SSD-VGG16": "#ec4899",
    "YOLO+FRCNN Ensemble": "#a855f7",
    "YOLOv8m+CBAM": "#22c55e",
    "YOLOv8m+CoordAttn": "#eab308",
    "RT-DETR": "#10b981",
}

print("Config loaded")
print(f"   CONF_THRESH  = {CONF_THRESH}")
print(f"   IOU_THRESH   = {IOU_THRESH}")
print(f"   IMG_SIZE     = {IMG_SIZE}")
print(f"   MAX_IMAGES   = {MAX_IMAGES}")
print(f"   SKIP_TRAINING= {SKIP_TRAINING}")

Config loaded
   CONF_THRESH  = 0.25
   IOU_THRESH   = 0.45
   IMG_SIZE     = 640
   MAX_IMAGES   = None
   SKIP_TRAINING= True


 Dataset Loading

In [5]:
# Downloads all three Kaggle pothole datasets through kagglehub.
import kagglehub

print("Downloading datasets via kagglehub ...")
path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")

DATASET_ROOTS = {
    "chitholian": path_1,
    "andrewmvd": path_2,
    "ashishkumar": path_3,
}
print(f"Dataset roots: {DATASET_ROOTS}")

Dataset roots: {'chitholian': '/home/vr3/.cache/kagglehub/datasets/chitholian/annotated-potholes-dataset/versions/1', 'andrewmvd': '/home/vr3/.cache/kagglehub/datasets/andrewmvd/pothole-detection/versions/1', 'ashishkumar': '/home/vr3/.cache/kagglehub/datasets/ashishkumarak/training-setzip/versions/1'}


In [6]:
# Defines the unified recursive XML annotation loader and loads the chitholian dataset.
def load_annotated_potholes(root, max_imgs=None):
    root = Path(root)
    records = []
    for img_path in list(root.rglob("*.jpg")) + list(root.rglob("*.png")):
        xml_path = img_path.with_suffix(".xml")
        if not xml_path.exists():
            xml_path = img_path.parent.parent / "annotations" / (img_path.stem + ".xml")
        gt_boxes = []
        if xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        {
                            "label": (obj.find("name").text or "pothole").lower(),
                            "xmin": float(bb.find("xmin").text),
                            "ymin": float(bb.find("ymin").text),
                            "xmax": float(bb.find("xmax").text),
                            "ymax": float(bb.find("ymax").text),
                        }
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
        if max_imgs is not None and len(records) >= max_imgs:
            break
    print(f"[Annotated Potholes] images={len(records)}")
    return records


records_1 = load_annotated_potholes(DATASET_ROOTS["chitholian"])


[Annotated Potholes] images=665


In [7]:
# Loads the andrewmvd dataset through the same unified loader.
records_2 = load_annotated_potholes(DATASET_ROOTS["andrewmvd"])


[Annotated Potholes] images=665


In [8]:
# Loads the ashishkumarak dataset through the CSV loader, since this source uses train/labels.csv rather than XML.
import pandas as pd


def load_ashishkumar_csv(root, max_imgs=None):
    root = Path(root)
    csv_path = root / "train" / "labels.csv"
    img_dir = root / "train" / "images"
    df = pd.read_csv(csv_path)
    grouped = df.groupby("ImageID")

    records = []
    img_paths = sorted(img_dir.glob("*.jpg"))[:max_imgs]
    for img_path in img_paths:
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    {
                        "label": "pothole",
                        "xmin": float(row["XMin"]),
                        "ymin": float(row["YMin"]),
                        "xmax": float(row["XMax"]),
                        "ymax": float(row["YMax"]),
                    }
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    print(
        f"[ashishkumar CSV] images={len(records)} ({sum(len(r['gt_boxes']) for r in records)} gt boxes)"
    )
    return records


records_3 = load_ashishkumar_csv(DATASET_ROOTS["ashishkumar"])


[ashishkumar CSV] images=674 (1371 gt boxes)


In [9]:
# Tags each record with its source, merges all three, removes duplicates by MD5 and then by normalized-pixel comparison, and defines the canonical train/val split.
for r in records_1:
    r["source"] = "annotated_dataset"
for r in records_2:
    r["source"] = "voc_dataset"
for r in records_3:
    r["source"] = "csv_dataset"

records = records_1 + records_2 + records_3
print(f"Total images before dedup: {len(records)}")


import numpy as np
from PIL import Image

NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0  # mean abs pixel diff (0-255 scale), true duplicates
# measured at 0.10-0.50, unrelated images much higher


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


annotated_only = [r for r in records if r["gt_boxes"]]
print("Computing normalized pixel arrays for dedup ...")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in annotated_only])

keep_mask = np.ones(len(annotated_only), dtype=bool)
seen_arrs = []
for i in range(len(annotated_only)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

n_before = len(records)
records = [r for r, keep in zip(annotated_only, keep_mask) if keep]
n_after = len(records)
print(f"Total images after dedup: {n_after}")
print(f"Duplicates removed: {n_before - n_after}")


import random as _random

_random.seed(42)
_annotated = [r for r in records if r["gt_boxes"]]
_shuffled = _annotated.copy()
_random.shuffle(_shuffled)
_split_idx = int(len(_shuffled) * 0.8)
train_recs = _shuffled[:_split_idx]
val_recs = _shuffled[_split_idx:]
print(
    f"\nCanonical split: train={len(train_recs)}  val={len(val_recs)}  (seed=42, shuffled)"
)

# Quick dataset summary
src_counts = defaultdict(int)
for r in records:
    src_counts[r["source"]] += 1
for src, cnt in src_counts.items():
    print(f"  {src}: {cnt} images")

Total images before dedup: 2004
Computing normalized pixel arrays for dedup ...
Total images after dedup: 926
Duplicates removed: 1078

Canonical split: train=740  val=186  (seed=42, shuffled)
  annotated_dataset: 665 images
  voc_dataset: 9 images
  csv_dataset: 252 images


In [10]:
# Reports the dataset composition after deduplication, images, boxes and per-source breakdown, as given in the paper's dataset table.

total_boxes = sum(len(r["gt_boxes"]) for r in records)
print(f"Final dataset: {len(records)} images, {total_boxes} ground-truth boxes")
print(f"Mean boxes per image: {total_boxes / len(records):.4f}")


from collections import defaultdict

per_source_imgs = defaultdict(int)
per_source_boxes = defaultdict(int)
for r in records:
    src = r.get("source", "unknown")
    per_source_imgs[src] += 1
    per_source_boxes[src] += len(r["gt_boxes"])
for src in per_source_imgs:
    print(f"  {src}: {per_source_imgs[src]} images, {per_source_boxes[src]} boxes")

Final dataset: 926 images, 2354 ground-truth boxes
Mean boxes per image: 2.5421
  annotated_dataset: 665 images, 1740 boxes
  voc_dataset: 9 images, 31 boxes
  csv_dataset: 252 images, 583 boxes


YOLO Format Conversion

In [11]:
# Converts the canonical split into YOLO-format image and label directories and writes data.yaml.
YOLO_DIR = os.path.join(ROOT, "yolo_dataset_rtdetr")
DATA_YAML = f"{YOLO_DIR}/data.yaml"

for split in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{YOLO_DIR}/{split}", exist_ok=True)


print(f"Using canonical split defined in Cell 12")
print(f"Train: {len(train_recs)}  |  Val: {len(val_recs)}")


def convert_to_yolo(rec_list, split):
    skipped = 0
    for rec in tqdm(rec_list, desc=f"  Converting {split}"):
        img = cv2.imread(str(rec["image_path"]))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]
        dst_img = f"{YOLO_DIR}/images/{split}/{rec['image_path'].name}"
        shutil.copy(str(rec["image_path"]), dst_img)
        dst_lbl = f"{YOLO_DIR}/labels/{split}/{rec['image_path'].stem}.txt"
        with open(dst_lbl, "w") as f:
            for g in rec["gt_boxes"]:
                cx = ((g["xmin"] + g["xmax"]) / 2) / w
                cy = ((g["ymin"] + g["ymax"]) / 2) / h
                bw = (g["xmax"] - g["xmin"]) / w
                bh = (g["ymax"] - g["ymin"]) / h
                f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
    print(f"  {split}: {len(rec_list) - skipped} written ({skipped} skipped)")


convert_to_yolo(train_recs, "train")
convert_to_yolo(val_recs, "val")

yaml_config = {
    "path": YOLO_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["pothole"],
}
with open(DATA_YAML, "w") as f:
    yaml.dump(yaml_config, f)
print(f"data.yaml saved to {DATA_YAML}")

Using canonical split defined in Cell 12
Train: 740  |  Val: 186


  Converting train:   0%|          | 0/740 [00:00<?, ?it/s]

  train: 740 written (0 skipped)


  Converting val:   0%|          | 0/186 [00:00<?, ?it/s]

  val: 186 written (0 skipped)
data.yaml saved to ./yolo_dataset_rtdetr/data.yaml


In [12]:
# Verifies the converted dataset by counting images and labels per split and previewing a sample label file.
for split in ["train", "val"]:
    imgs = glob.glob(f"{YOLO_DIR}/images/{split}/*")
    lbls = glob.glob(f"{YOLO_DIR}/labels/{split}/*")
    print(f"  {split:5s} -- images: {len(imgs):4d}  |  labels: {len(lbls):4d}")

# Preview a sample label file
sample_lbl = glob.glob(f"{YOLO_DIR}/labels/train/*.txt")
if sample_lbl:
    with open(sample_lbl[0]) as f:
        lines = f.readlines()
    print(f"\nSample label ({Path(sample_lbl[0]).name}):")
    for line in lines[:3]:
        print(f"  {line.strip()}")

print("\nYOLO dataset verified")

  train -- images:  740  |  labels:  740
  val   -- images:  186  |  labels:  186

Sample label (img-294.txt):
  0 0.731273 0.906667 0.147940 0.133333
  0 0.526217 0.823333 0.067416 0.060000
  0 0.382022 0.655000 0.029963 0.036667

YOLO dataset verified


 Metric Helpers

In [13]:
# Defines box_iou, compute_ap, evaluate, the confidence sweep and the throughput helper, the shared scoring functions used for RT-DETR.
import numpy as np
from collections import defaultdict


def box_iou(b1, b2):
    ix1 = max(b1[0], b2[0])
    iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2])
    iy2 = min(b1[3], b2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    u = a1 + a2 - inter
    return inter / u if u > 0 else 0.0


def compute_ap_voc11(recalls, precisions):
    """Legacy PASCAL VOC-2007 11-point AP. Kept for continuity only --
    do NOT report it alongside Ultralytics model.val() figures."""
    ap = 0.0
    for thr in np.linspace(0, 1, 11):
        ps = [p for r, p in zip(recalls, precisions) if r >= thr]
        ap += max(ps) if ps else 0.0
    return ap / 11


def compute_ap(recalls, precisions):
    # COCO-style 101-point interpolated AP
    if len(recalls) == 0:
        return 0.0
    mrec = np.concatenate(([0.0], np.asarray(recalls, dtype=float), [1.0]))
    mpre = np.concatenate(([1.0], np.asarray(precisions, dtype=float), [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))  # precision envelope
    x = np.linspace(0, 1, 101)
    _trapz = getattr(np, "trapezoid", None) or np.trapz  # numpy>=2 rename
    return float(_trapz(np.interp(x, mrec, mpre), x))


def evaluate(all_gt, all_preds, iou_thr=0.5):
    # Greedy score-ordered matching, one detection per ground-truth box.
    pl = []
    for idx, preds in enumerate(all_preds):
        for p in preds:
            pl.append((idx, p["conf"], [p["xmin"], p["ymin"], p["xmax"], p["ymax"]]))
    pl.sort(key=lambda x: -x[1])

    num_gt = sum(len(g) for g in all_gt)
    matched = defaultdict(set)
    TP = []
    FP = []
    for idx, conf, pb in pl:
        gts = all_gt[idx]
        taken = matched[idx]
        best_iou, best_j = 0.0, -1
        for j, g in enumerate(gts):
            if j in taken:  # <-- the fix
                continue
            iou = box_iou(pb, [g["xmin"], g["ymin"], g["xmax"], g["ymax"]])
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thr and best_j >= 0:
            TP.append(1)
            FP.append(0)
            taken.add(best_j)
        else:
            TP.append(0)
            FP.append(1)

    tp_c = np.cumsum(TP)
    fp_c = np.cumsum(FP)
    recs = (tp_c / num_gt).tolist() if num_gt > 0 else [0.0]
    precs = (tp_c / np.maximum(tp_c + fp_c, 1e-12)).tolist()
    if not pl or num_gt == 0:
        ap_coco = ap_voc = 0.0
    else:
        ap_coco = compute_ap(recs, precs)
        ap_voc = compute_ap_voc11(recs, precs)

    ttp = int(sum(TP))
    tfp = int(sum(FP))
    tfn = num_gt - ttp
    pr = ttp / (ttp + tfp) if (ttp + tfp) > 0 else 0.0
    rc = ttp / (ttp + tfn) if (ttp + tfn) > 0 else 0.0
    f1 = 2 * pr * rc / (pr + rc) if (pr + rc) > 0 else 0.0
    return {
        "mAP@0.5": round(ap_coco, 4),
        "mAP@0.5(VOC11)": round(ap_voc, 4),
        "Precision": round(pr, 4),
        "Recall": round(rc, 4),
        "F1": round(f1, 4),
        "TP": ttp,
        "FP": tfp,
        "FN": tfn,
    }


CONF_SWEEP = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]


def threshold_preds(all_preds, conf):
    return [[p for p in preds if p["conf"] >= conf] for preds in all_preds]


def sweep_best_conf(all_gt, all_preds, conf_grid=None, iou_thr=0.5, verbose=True):
    conf_grid = conf_grid if conf_grid is not None else CONF_SWEEP
    rows = []
    for c in conf_grid:
        m = evaluate(all_gt, threshold_preds(all_preds, c), iou_thr=iou_thr)
        rows.append(
            {
                "conf": c,
                **{k: m[k] for k in ("Precision", "Recall", "F1", "TP", "FP", "FN")},
            }
        )
    df_sweep = pd.DataFrame(rows).set_index("conf")
    best_conf = float(df_sweep["F1"].idxmax())
    best_m = evaluate(all_gt, threshold_preds(all_preds, best_conf), iou_thr=iou_thr)
    if verbose:
        print(
            f"     conf sweep -> best F1={best_m['F1']:.4f} at conf={best_conf:.2f} "
            f"(P={best_m['Precision']:.4f}  R={best_m['Recall']:.4f})"
        )
    return best_conf, best_m, df_sweep


def summarize(all_gt, all_preds, fps_times=None, iou_thr=0.5, verbose=True):
    """Build the result dict from UNTHRESHOLDED predictions (conf=0.001)."""
    m_map = evaluate(all_gt, all_preds, iou_thr=iou_thr)
    m_def = evaluate(all_gt, threshold_preds(all_preds, CONF_THRESH), iou_thr=iou_thr)
    best_conf, m_best, df_sweep = sweep_best_conf(
        all_gt, all_preds, iou_thr=iou_thr, verbose=verbose
    )
    out = {
        "mAP@0.5": m_map["mAP@0.5"],
        "Precision": m_def["Precision"],
        "Recall": m_def["Recall"],
        "F1": m_def["F1"],
        "TP": m_def["TP"],
        "FP": m_def["FP"],
        "FN": m_def["FN"],
        "best_conf": best_conf,
        "P@best": m_best["Precision"],
        "R@best": m_best["Recall"],
        "F1@best": m_best["F1"],
    }
    if fps_times:
        out["FPS"] = fps_from_times(fps_times)
    out["_sweep"] = df_sweep
    return out


# FPS measurement
FPS_WARMUP = 5


def _sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def fps_from_times(times, warmup=FPS_WARMUP):
    t = [x for x in times[warmup:] if x > 0]
    if not t:
        t = [x for x in times if x > 0]
    return round(len(t) / sum(t), 1) if t else 0.0


# Timing warm-up
WARMUP_ITERS = 10


def warmup(forward, n=WARMUP_ITERS):
    """Run `forward` n times untimed, then synchronise."""
    for _ in range(n):
        forward()
    _sync()


def first_image(recs):
    """First decodable image in `recs`, or None."""
    for rec in recs:
        img = cv2.imread(str(rec["image_path"]))
        if img is not None:
            return img
    return None


print("Metric helpers ready (COCO-101 AP, fixed matcher, conf sweep, throughput FPS)")


Metric helpers ready (COCO-101 AP, fixed matcher, conf sweep, throughput FPS)


In [14]:
# Sanity-checks the metric helpers against hand-computed cases before any model is scored.
_b1, _b2 = [0, 0, 10, 10], [5, 5, 15, 15]
_expected = 25 / (100 + 100 - 25)
_got = box_iou(_b1, _b2)
assert abs(_got - _expected) < 1e-4, f"box_iou failed: {_got} vs {_expected}"
print(f"  box_iou([0,0,10,10], [5,5,15,15]) = {_got:.4f}")

_ap = compute_ap([0.0, 0.5, 1.0], [1.0, 1.0, 1.0])
assert _ap > 0.99, f"compute_ap perfect failed: {_ap}"
print(
    f"  compute_ap COCO-101 (perfect)     = {_ap:.4f}   (0.995 is the reference ceiling)"
)
_apv = compute_ap_voc11([0.0, 0.5, 1.0], [1.0, 1.0, 1.0])
print(f"  compute_ap VOC-11  (perfect)      = {_apv:.4f} ")

# Matcher: two overlapping GTs must both be matched, not one TP + one FP
_gt2 = [
    [
        {"xmin": 0, "ymin": 0, "xmax": 100, "ymax": 100},
        {"xmin": 10, "ymin": 10, "xmax": 110, "ymax": 110},
    ]
]
_pr2 = [
    [
        {"conf": 0.9, "xmin": 5, "ymin": 5, "xmax": 105, "ymax": 105},
        {"conf": 0.8, "xmin": -5, "ymin": -5, "xmax": 95, "ymax": 95},
    ]
]
_m2 = evaluate(_gt2, _pr2)
assert (_m2["TP"], _m2["FP"], _m2["FN"]) == (2, 0, 0), _m2
print(
    f"  matcher (2 overlapping GTs)       = TP={_m2['TP']} FP={_m2['FP']} FN={_m2['FN']}  "
)

# Degenerate curve: no predictions must score 0.0, not 0.5
_m3 = evaluate([[{"xmin": 0, "ymin": 0, "xmax": 10, "ymax": 10}]], [[]])
assert _m3["mAP@0.5"] == 0.0, _m3
print(f"  empty predictions                 = mAP {_m3['mAP@0.5']}  ")

_gt = [[{"xmin": 0, "ymin": 0, "xmax": 10, "ymax": 10}]]
_preds = [
    [{"conf": 0.9, "xmin": 1, "ymin": 1, "xmax": 11, "ymax": 11, "label": "pothole"}]
]
_m = evaluate(_gt, _preds)
print(
    f"  evaluate (single match)           = P={_m['Precision']} R={_m['Recall']} F1={_m['F1']}  "
)

print("\n All metric sanity checks passed")


  box_iou([0,0,10,10], [5,5,15,15]) = 0.1429
  compute_ap COCO-101 (perfect)     = 0.9950   (0.995 is the reference ceiling)
  compute_ap VOC-11  (perfect)      = 1.0000 
  matcher (2 overlapping GTs)       = TP=2 FP=0 FN=0  
  empty predictions                 = mAP 0.0  
  evaluate (single match)           = P=1.0 R=1.0 F1=1.0  

 All metric sanity checks passed
